# message的使用

## 1、消息格式

举例：json格式

In [1]:

from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

# 加载配置文件
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 获取大模型
model = init_chat_model(
    # model="deepseek-v4-flash",
    # model_provider="deepseek",
    # 或者
    model="deepseek:deepseek-flash",
    temperature=1.9,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    max_tokens=10000,
)


In [2]:
#json格式的消息
messages = [
    {"role":"system","content":"你是一个友好的AI助手"},
    {"role":"user","content":"1 + 2 = ?"},
    {"role":"assistant","content":"3"},
    {"role":"user","content":"我刚才问了什么问题？"},
]

response= model.invoke(messages)
print(response)

content='你刚才问的是：**“1 + 2 = ?”**' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。用户问“我刚才问了什么问题？” 需要指出上一个问题是“1 + 2 = ?”。简洁。'} response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 50, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 26, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 50}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '7b2cd2b5-3b76-41ec-9dcd-1ee06477e356', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a19700-621d-703f-9871-458e94ae4878-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 50, 'output_tokens': 42, 'total_tokens': 92, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 26}}


举例2：消息对象列表

In [4]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage

#json格式的消息
messages = [
    SystemMessage(content="你是一个友好的AI助手"),
    HumanMessage("1 + 2 = ?"),
    AIMessage("3"),
    HumanMessage("我刚才问了什么问题？"),
]

response= model.invoke(messages)
print(response)

content='你刚才问的是：“1 + 2 = ?”' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户中文问题：“我刚才问了什么问题？” 需要回忆对话：用户先问“1 + 2 = ?”，助手答“3”。然后用户问“我刚才问了什么问题？” 所以回答：你刚才问的是“1 + 2 = ?”。可以简洁。注意是“刚才”可能指上一轮，即问“1+2=?”。 回答中文。'} response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 50, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 81, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 50}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '7549ebcc-3a97-476d-918f-c514917ac866', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a19702-c82f-710f-923c-5f4e112598a1-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 50, 'output_tokens': 94, 'total_tokens': 144, 'input_token_details

## 2、HumanMessage的使用

举例1：使用CloseAI平台

In [5]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

messages = [
    SystemMessage("你是一个信息抽取器。你会收到多条来自不同发言者的 user 消息。每条消息可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出 JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出格式：{\"speakers\":[{\"name\":\"...\",\"claim\":\"...\"}]}"),
    HumanMessage(
        content="我认为 1+1=2",
        name="Bob"
    ),
    HumanMessage(
        content="我认为 1+1>2",
        name="Tom"
    ),
    HumanMessage(
        content="请列出谁说了什么，不要判断对错。",
        name="audience"
    )
]

response = model.invoke(messages)
print(response.content)

{"speakers":[{"name":"Bob","claim":"我认为 1+1=2"},{"name":"Tom","claim":"我认为 1+1>2"},{"name":"audience","claim":"请列出谁说了什么，不要判断对错。"}]}


## 3、AIMessage的使用

举例

In [6]:
from langchain_core.messages import SystemMessage,HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

messages = [
    SystemMessage("你叫小智，是一名助人为乐的助手。"),
    HumanMessage("你好，好久不见，请介绍下你自己。")
]

response = model.invoke(messages)
rprint(response)


AIMessage(
    content='你好，好久不见！我是小智，一名助人为乐的助手，很高兴再次见到你。\n\n我可以帮你做很多事，比如：\n- 
回答问题、解释概念\n- 写作润色、起草邮件/文案\n- 翻译、总结、整理信息\n- 头脑风暴、提供方案和建议\n- 
协助学习、做题讲解\n\n如果你愿意，也可以直接告诉我你现在想做什么，我马上帮你。',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 110,
            'prompt_tokens': 34,
            'total_tokens': 144,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
            'latency_checkpoint': {
                'engine_tbt_ms': 5,
                'engine_ttft_ms': 39,
                'engine_ttlt_ms': 561,
                'pre_inference_ms': 79,
                'service_tbt_ms': 5,
                'service_ttft_ms': 502,
                'service_ttlt_ms': 1019,
                'user_visible_ttft_ms': 423
            }
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5.4-mini-2026-03-17',
        'system_fingerprint': None,
        'id': 'chatcmpl-EfOywkdcmtgNfSUpL6V8IF1jJ2L8x',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a19716-54ea-766c-b42a-5011c332f627-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 34,
        'output_tokens': 110,
        'total_tokens': 144,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

## 4、ToolMessage的使用

举例1：json格式

In [7]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

def get_weather(city: str) -> str:
    return "不错哦~"

# 模拟模型绑定工具
# model_with_tools = model.bind_tools([get_weather])

ai_message = {
    "role": "assistant",
    "content": "",
    "tool_calls": [{
        "name": "get_weather",
        "args": {"location": "北京"},
        "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
    }]
}

tool_message = {
    "role": "tool",
    "content": "今天北京天气晴朗，万里无云~",
    "tool_call_id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
}

messages = [
    {"role": "user", "content": "北京天气如何"},
    ai_message,
    tool_message
]

response = model.invoke(messages)

print(response)

content='今天北京天气晴朗，万里无云。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 48, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 5, 'engine_ttft_ms': 50, 'engine_ttlt_ms': 122, 'pre_inference_ms': 65, 'service_tbt_ms': 2, 'service_ttft_ms': 549, 'service_ttlt_ms': 572, 'user_visible_ttft_ms': 484}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EfP1dPF2lnTsmq4FiBluyJFx34vb1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a19718-e0ba-7496-8d8b-b75bd1cb726b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 48, 'output_tokens': 14, 'total_tokens': 62, 'input_token_details': {'audio': 0, 'cach

举例2：消息对象格式

In [8]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from langchain_core.messages.tool import ToolMessage
# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

def get_weather(city: str) -> str:
    return "不错哦~"

# 模拟模型绑定工具
# model_with_tools = model.bind_tools([get_weather])

# ai_message = {
#     "role": "assistant",
#     "content": "",
#     "tool_calls": [{
#         "name": "get_weather",
#         "args": {"location": "北京"},
#         "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
#     }]
# }

ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "北京"},
        "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
    }]
)

# tool_message = {
#     "role": "tool",
#     "content": "今天北京天气晴朗，万里无云~",
#     "tool_call_id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
# }

tool_message = ToolMessage(
    content="今天北京天气晴朗，万里无云~",
    tool_call_id = "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
)

messages = [
    # {"role": "user", "content": "北京天气如何"},
    HumanMessage(content="北京天气如何"),
    ai_message,
    tool_message
]

response = model.invoke(messages)

print(response)

content='今天北京天气晴朗，万里无云。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 48, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 5, 'engine_ttft_ms': 50, 'engine_ttlt_ms': 122, 'pre_inference_ms': 74, 'service_tbt_ms': 5, 'service_ttft_ms': 500, 'service_ttlt_ms': 569, 'user_visible_ttft_ms': 426}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EfP4HR4UgakHfqgxB2qrnwpTuPnoT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1971b-61ef-71fc-adb3-7e6e53062bfe-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 48, 'output_tokens': 14, 'total_tokens': 62, 'input_token_details': {'audio': 0, 'cach

## 5、对话历史优化
举例：

In [10]:
def keep_recent_messages(messages,max_pairs = 3):
    """
    保留最近的N轮对话
    max_pairs : 保留对话的轮数 （每轮 = user + assistant）
    """

    # 分离system 消息和对话消息
    system_messages = [m for m in messages if m.get("role") == "system"]
    conversation_messages = [m for m in messages if m.get("role") != "system"]

    # 只保留最近的消息对
    recent_messages = conversation_messages[-(max_pairs * 2):]

    # 返回系统消息和最近的消息对
    return system_messages + recent_messages

In [11]:
# 初始化
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]

# 第 1 轮
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})

# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})

# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})

print(f"原始消息数: {len(long_conversation)}")  # 7

# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)

print(f"优化后消息数: {len(optimized)}")  # 5
print(f"保留的内容: system + 最近2轮对话")

# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么？"})
# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

原始消息数: 7
优化后消息数: 5
保留的内容: system + 最近2轮对话

AI 回复: 你第一个问题问的是：**“列表和元组有什么区别？用一句解释”**


## 6、多轮对话聊天机器人

In [13]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# 1. 基础配置
MODEL_NAME = "gpt-5.4-mini"
MAX_PAIRS_HISTORY = 10
EXIT_WORD = "quit"

# 2. 初始化模型
model = init_chat_model(
    model=MODEL_NAME,
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

# 3. 维护一个消息列表
messages = [
    {
        "role": "system",
        "content":"你是小谷姐姐，是尚硅谷教育的数字员工，也是一名耐心、友好的AI助手，可以回答学的问题"
    }
]

print(f"请输入具体的问题，当输入{EXIT_WORD}的时候，结束对话")

i = 1 # 描述对话的轮数
while True:
    print("\n","="*10,f"第{i}轮对话开始","="*10,"\n")

    user_input = input("请输入：")
    print(user_input,"\n",end="",flush=True,)

    # 判断是否结束当前会话
    if user_input == EXIT_WORD:
        print("会话已结束，欢迎下次再来")
        break

    # 4. 将用户的信息添加到消息列表中
    messages.append({
        "role": "user",
        "content": user_input
    })

    print("小谷姐姐：",end="",flush=True)

    # 5. 拼接AI回复的消息信息
    reply_content = ""

    #6. 优化历史记忆
    memory_messages = keep_recent_messages(messages, max_pairs=MAX_PAIRS_HISTORY)

    # 7. 流式输出模型的响应
    for chunk in model.stream(memory_messages):
        if chunk.content:
            print(chunk.content,end="",flush=True)
            reply_content += chunk.content

    print("\n","="*10,f"第{i}轮对话结束","="*10,"\n")

    i += 1

    # 8. 将模型的响应添加到消息列表
    messages.append({"role":"assistant", "content": reply_content})

请输入具体的问题，当输入quit的时候，结束对话

 ========== 第1轮对话开始 ========== 

hello小谷姐姐：Hello！我是小谷姐姐，很高兴认识你 😊  
有什么学习上的问题，或者需要我帮你解答的吗？
 ========== 第1轮对话结束 ========== 


 ========== 第2轮对话开始 ========== 



KeyboardInterrupt: Interrupted by user